In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kendalltau
import re  # For regex-based string parsing
import numpy as np
from scipy.stats import wasserstein_distance, spearmanr, kendalltau

df = pd.read_csv('Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv')
bertDf = pd.read_csv('Results/Classification/bert.csv')
robertaDf = pd.read_csv('Results/Classification/llama.csv')
llamaDf = pd.read_csv('Results/Classification/roberta.csv')
mistralDf = pd.read_csv('Results/Classification/mistral/24B_results.csv')



In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

# Use the PGF backend
matplotlib.use("pgf")

# Configure LaTeX rendering
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",  # or "lualatex", "xelatex"
    "text.usetex": True,
    "font.family": "serif",  # Use LaTeX default serif font
    "font.serif": [],        # Let LaTeX choose the font
    "axes.labelsize": 10,    # LaTeX font sizes
    "font.size": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "pgf.rcfonts": False,    # Do not override LaTeX settings
})

In [3]:
df

,url,title,text,authors,domain,score,country,language,topics,paywall,opinion_advocacy,label
0,https://nemontananews.com/stories/670532029-1-...,1 gas station in cities throughout Phillips Co...,Federal Energy Regulatory Commission (FERC) Ch...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
1,https://nemontananews.com/stories/670175501-mc...,McCone County: Income Maintenance dependency r...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
2,https://nemontananews.com/stories/670489322-da...,"Daniels County residents received $13,488 per ...","John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
3,https://nemontananews.com/stories/670175454-da...,Daniels County: Income Maintenance made up 0.8...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
4,https://nemontananews.com/stories/670413125-wh...,Where could drivers find the cheapest gas in c...,Federal Energy Regulatory Commission (FERC) Co...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...
19471,https://www.truthdig.com/articles/us-house-vot...,US House Vote Rolls Back Wetlands Regulations,The U.S. House voted Thursday to undo a Biden ...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19472,https://www.truthdig.com/articles/environmenta...,Environmental Racism in Canada,Informed nine months after the fact that its h...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19473,https://www.truthdig.com/articles/the-runaway-...,The Runaway Train of Corporate Greed,Thursday’s U.S. Senate hearing about the ongoi...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19474,https://www.truthdig.com/articles/ron-desantis...,Ron DeSantis’ Phantom War Against Voter Fraud,A new Florida law gives Gov.Ron DeSantis’ pros...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3


# Calcolo l'accuracy pesata prendendo spunto da questo paper

### wconf: The weighted confusion matrix and accuracy scores package for R
### DOI: 10.13140/RG.2.2.15992.83207

*testo in corsivo*![image.png](attachment:2bf9c679-45dc-4569-87a2-32cfb90ef1e9.png)



```
# Questo è formattato come codice
```

## Utilizo come peso il 50% per le classi adiacenti

In [4]:
import re
import numpy as np
import pandas as pd # Assuming pandas is used for DataFrames

def compute_weighted_confusion(df):
    """
    Adds weighted accuracies (0.5 and 1.0 adjacent weights) to the dataframe
    based on the confusion matrices, as intended in:
    https://www.researchgate.net/publication/376392764
    """
    weighted_accuracies_0_5 = []
    weighted_accuracies_1_0 = []

    # Loop over the CMs of the various folds
    for confusion_str in df['Confusion_Matrix']:
        # Extract the numbers and cast them to integers
        numbers = re.findall(r'\d+', confusion_str)
        cm_flat = list(map(int, numbers))

        # Understand the number of classes
        n = int(np.sqrt(len(cm_flat))) # using the square root of the matrix length

        # Transform the array of numbers into a confusion matrix for numpy
        cm = np.array(cm_flat).reshape(n, n)

        # Calculate the total number of predictions (denominator)
        total = np.sum(cm)

        # --- Calculate Weighted Accuracy with adjacent_weight = 0.5 ---
        weights_0_5 = np.zeros((n, n))
        for i in range(n):
            weights_0_5[i, i] = 1.0  # correct class -> weight == 1
            if i > 0:
                weights_0_5[i, i-1] = 0.5  # Adjacent class -> weight == 0.5
            if i < n-1:
                weights_0_5[i, i+1] = 0.5  # Adjacent class -> weight == 0.5

        # Apply the weights and sum to get the weighted correct predictions (numerator)
        weighted_correct_0_5 = np.sum(cm * weights_0_5)

        # Calculate the accuracy for 0.5 weight
        accuracy_0_5 = (weighted_correct_0_5 / total) if total > 0 else 0
        weighted_accuracies_0_5.append(accuracy_0_5)

        # --- Calculate Weighted Accuracy with adjacent_weight = 1.0 ---
        weights_1_0 = np.zeros((n, n))
        for i in range(n):
            weights_1_0[i, i] = 1.0  # correct class -> weight == 1
            if i > 0:
                weights_1_0[i, i-1] = 1.0  # Adjacent class -> weight == 1.0
            if i < n-1:
                weights_1_0[i, i+1] = 1.0  # Adjacent class -> weight == 1.0

        # Apply the weights and sum to get the weighted correct predictions (numerator)
        weighted_correct_1_0 = np.sum(cm * weights_1_0)

        # Calculate the accuracy for 1.0 weight
        # Note: If total is 0 (empty confusion matrix), accuracy is 0.
        accuracy_1_0 = (weighted_correct_1_0 / total) if total > 0 else 0
        weighted_accuracies_1_0.append(accuracy_1_0)


    # Add new columns to DataFrame
    df['Weighted_Accuracy_0.5'] = weighted_accuracies_0_5
    df['Weighted_Accuracy_1.0'] = weighted_accuracies_1_0

    return df


bertDf = compute_weighted_confusion(bertDf)
robertaDf = compute_weighted_confusion(robertaDf)
llamaDf = compute_weighted_confusion(llamaDf)
mistralDf = compute_weighted_confusion(mistralDf)

In [5]:
robertaDf['ModelName']='RobertaLarge'
robertaDf

,Fold,Train_Losses_Per_Epoch,Val_Losses_Per_Epoch,Balanced_Accuracy,Accuracy,Precision_Micro,Precision_Macro,Precision_Weighted,Recall_Micro,Recall_Macro,Recall_Weighted,F1_Micro,F1_Macro,F1_Weighted,Confusion_Matrix,Val_True_Labels,Val_Predictions,Weighted_Accuracy_0.5,Weighted_Accuracy_1.0,ModelName
0,1,"[2.273344498424704, 0.6563329227557215, 0.3964...","[0.9109003281593323, 0.523340392112732, 0.4629...",0.546540,0.614634,0.614634,0.561393,0.617791,0.614634,0.546540,0.614634,0.614634,0.551343,0.612938,"[[776, 22, 11, 90, 18], [28, 397, 186, 142, 43...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.728755,0.842875,RobertaLarge
1,2,"[2.2537926341394128, 0.6427574809383138, 0.359...","[0.8524569582939148, 0.4681211179494858, 0.401...",0.542944,0.573264,0.573264,0.545999,0.610328,0.573264,0.542944,0.573264,0.573264,0.536701,0.584107,"[[729, 137, 56, 37, 28], [7, 400, 112, 49, 32]...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, ...",0.721399,0.869533,RobertaLarge
2,3,"[2.302797350053037, 0.6249480824122268, 0.3466...","[0.9556651520729065, 0.5182030880451203, 0.462...",0.521130,0.580355,0.580355,0.527020,0.605529,0.580355,0.521130,0.580355,0.580355,0.520926,0.590047,"[[840, 12, 18, 0, 11], [18, 247, 119, 78, 76],...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[4, 4, 4, 1, 3, 3, 4, 3, 1, 3, 3, 1, 3, 4, 1, ...",0.718554,0.856753,RobertaLarge
3,4,"[2.351019258719099, 0.6279265853769416, 0.3830...","[0.8965884923934937, 0.5370388412475586, 0.471...",0.561900,0.591779,0.591779,0.582187,0.632612,0.591779,0.561900,0.591779,0.591779,0.567835,0.606555,"[[792, 162, 26, 95, 17], [11, 190, 12, 340, 55...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.684850,0.777921,RobertaLarge
4,5,"[2.4721066037175854, 0.5970372030118464, 0.338...","[1.0018611645698547, 0.47080618143081665, 0.40...",0.569448,0.637091,0.637091,0.566040,0.637627,0.637091,0.569448,0.637091,0.637091,0.565431,0.634783,"[[946, 14, 24, 10, 0], [65, 175, 39, 104, 91],...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.738940,0.840790,RobertaLarge


In [6]:
llamaDf['ModelName']='LLama3'

llamaDf

,Fold,Balanced_Accuracy,Accuracy,Precision_Macro,Precision_Weighted,Recall_Macro,Recall_Weighted,F1_Macro,F1_Weighted,Confusion_Matrix,Val_True_Labels,Val_Predictions,Weighted_Accuracy_0.5,Weighted_Accuracy_1.0,ModelName
0,1,0.545132,0.607445,0.554182,0.613016,0.545132,0.607445,0.548338,0.608790,"[[770, 33, 7, 86, 21], [11, 391, 245, 115, 34]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.725417,0.843389,LLama3
1,2,0.533799,0.588337,0.538660,0.595666,0.533799,0.588337,0.532931,0.589032,"[[737, 80, 100, 53, 17], [11, 376, 140, 57, 16...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 1, 3, 1, 1, 3, ...",0.716580,0.844823,LLama3
2,3,0.533025,0.593179,0.528791,0.590510,0.533025,0.593179,0.526597,0.588478,"[[846, 14, 8, 10, 3], [38, 252, 107, 113, 28],...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0, 3, 4, 1, 3, 2, 4, 2, 2, 3, 3, 0, 0, 4, 2, ...",0.718827,0.844475,LLama3
3,4,0.548994,0.601344,0.556752,0.601384,0.548994,0.601344,0.549386,0.598390,"[[899, 75, 10, 102, 6], [44, 160, 35, 356, 13]...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.684592,0.767839,LLama3
4,5,0.540668,0.626093,0.534656,0.605368,0.540668,0.626093,0.535598,0.613762,"[[937, 10, 27, 17, 3], [86, 142, 53, 125, 68],...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.738190,0.850287,LLama3


In [7]:
bertDf['ModelName']='Bert'

bertDf

,Fold,Balanced_Accuracy,Accuracy,Precision_Micro,Precision_Macro,Precision_Weighted,Recall_Micro,Recall_Macro,Recall_Weighted,F1_Micro,F1_Macro,F1_Weighted,Confusion_Matrix,True_Labels,Predicted_Labels,Weighted_Accuracy_0.5,Weighted_Accuracy_1.0,ModelName
0,1,0.510007,0.570732,0.570732,0.519026,0.576988,0.570732,0.510007,0.570732,0.570732,0.510744,0.571108,"[[769, 37, 26, 68, 17], [30, 371, 275, 99, 21]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.694352,0.817972,Bert
1,2,0.487459,0.547319,0.547319,0.493514,0.560459,0.547319,0.487459,0.547319,0.547319,0.487836,0.551056,"[[727, 97, 69, 76, 18], [11, 299, 135, 123, 32...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",0.676674,0.806029,Bert
2,3,0.516695,0.604638,0.604638,0.534950,0.608693,0.604638,0.516695,0.604638,0.604638,0.515509,0.596506,"[[842, 5, 9, 15, 10], [14, 154, 84, 240, 46], ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3, 3, ...",0.734925,0.865211,Bert
3,4,0.531071,0.576008,0.576008,0.548393,0.597615,0.576008,0.531071,0.576008,0.576008,0.535750,0.581898,"[[777, 127, 11, 132, 45], [56, 147, 13, 351, 4...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.670760,0.765512,Bert
4,5,0.536891,0.616596,0.616596,0.542916,0.610219,0.616596,0.536891,0.616596,0.616596,0.537073,0.611042,"[[923, 7, 22, 35, 7], [82, 114, 42, 154, 82], ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.733567,0.850537,Bert


In [8]:
mistralDf['ModelName']='Mistral'

mistralDf

,Fold,Flash_Attention,Precision_Macro,Recall_Macro,F1_Macro,Precision_micro,Recall_micro,F1_micro,Precision_weighted,Recall_weighted,...,Balanced_Accuracy,MCC,Cohen_Kappa,Confusion_Matrix,Val_Article_IDs,Val_True_Labels,Val_Predictions,Weighted_Accuracy_0.5,Weighted_Accuracy_1.0,ModelName
0,1,True,0.596051,0.561743,0.568535,0.625417,0.625417,0.625417,0.665317,0.625417,...,0.561743,0.522586,0.518896,"[[768, 9, 31, 96, 13], [15, 374, 214, 96, 97],...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.730552,0.835687,Mistral
1,2,True,0.551455,0.548730,0.536507,0.605387,0.605387,0.605387,0.638654,0.605387,...,0.548730,0.501691,0.495725,"[[724, 191, 38, 24, 10], [3, 455, 96, 40, 6], ...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, ...",0.729553,0.853719,Mistral
2,3,True,0.542132,0.539485,0.534869,0.600273,0.600273,0.600273,0.599504,0.600273,...,0.539485,0.487755,0.486125,"[[831, 19, 22, 4, 5], [21, 277, 102, 112, 26],...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[4, 4, 4, 1, 1, 3, 1, 3, 3, 3, 3, 4, 1, 4, 4, ...",0.721010,0.841746,Mistral
3,4,True,0.554403,0.530380,0.520401,0.591779,0.591779,0.591779,0.581802,0.591779,...,0.530380,0.478604,0.471742,"[[944, 32, 48, 63, 5], [54, 111, 92, 345, 6], ...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.702042,0.812306,Mistral
4,5,True,0.564328,0.566713,0.559403,0.634091,0.634091,0.634091,0.630422,0.634091,...,0.566713,0.530551,0.526325,"[[963, 5, 17, 4, 5], [114, 151, 51, 72, 86], [...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.731692,0.829293,Mistral


In [9]:
bertDf

,Fold,Balanced_Accuracy,Accuracy,Precision_Micro,Precision_Macro,Precision_Weighted,Recall_Micro,Recall_Macro,Recall_Weighted,F1_Micro,F1_Macro,F1_Weighted,Confusion_Matrix,True_Labels,Predicted_Labels,Weighted_Accuracy_0.5,Weighted_Accuracy_1.0,ModelName
0,1,0.510007,0.570732,0.570732,0.519026,0.576988,0.570732,0.510007,0.570732,0.570732,0.510744,0.571108,"[[769, 37, 26, 68, 17], [30, 371, 275, 99, 21]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.694352,0.817972,Bert
1,2,0.487459,0.547319,0.547319,0.493514,0.560459,0.547319,0.487459,0.547319,0.547319,0.487836,0.551056,"[[727, 97, 69, 76, 18], [11, 299, 135, 123, 32...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",0.676674,0.806029,Bert
2,3,0.516695,0.604638,0.604638,0.534950,0.608693,0.604638,0.516695,0.604638,0.604638,0.515509,0.596506,"[[842, 5, 9, 15, 10], [14, 154, 84, 240, 46], ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3, 3, ...",0.734925,0.865211,Bert
3,4,0.531071,0.576008,0.576008,0.548393,0.597615,0.576008,0.531071,0.576008,0.576008,0.535750,0.581898,"[[777, 127, 11, 132, 45], [56, 147, 13, 351, 4...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.670760,0.765512,Bert
4,5,0.536891,0.616596,0.616596,0.542916,0.610219,0.616596,0.536891,0.616596,0.616596,0.537073,0.611042,"[[923, 7, 22, 35, 7], [82, 114, 42, 154, 82], ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.733567,0.850537,Bert


In [10]:
llamaDf

,Fold,Balanced_Accuracy,Accuracy,Precision_Macro,Precision_Weighted,Recall_Macro,Recall_Weighted,F1_Macro,F1_Weighted,Confusion_Matrix,Val_True_Labels,Val_Predictions,Weighted_Accuracy_0.5,Weighted_Accuracy_1.0,ModelName
0,1,0.545132,0.607445,0.554182,0.613016,0.545132,0.607445,0.548338,0.608790,"[[770, 33, 7, 86, 21], [11, 391, 245, 115, 34]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.725417,0.843389,LLama3
1,2,0.533799,0.588337,0.538660,0.595666,0.533799,0.588337,0.532931,0.589032,"[[737, 80, 100, 53, 17], [11, 376, 140, 57, 16...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 1, 3, 1, 1, 3, ...",0.716580,0.844823,LLama3
2,3,0.533025,0.593179,0.528791,0.590510,0.533025,0.593179,0.526597,0.588478,"[[846, 14, 8, 10, 3], [38, 252, 107, 113, 28],...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0, 3, 4, 1, 3, 2, 4, 2, 2, 3, 3, 0, 0, 4, 2, ...",0.718827,0.844475,LLama3
3,4,0.548994,0.601344,0.556752,0.601384,0.548994,0.601344,0.549386,0.598390,"[[899, 75, 10, 102, 6], [44, 160, 35, 356, 13]...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.684592,0.767839,LLama3
4,5,0.540668,0.626093,0.534656,0.605368,0.540668,0.626093,0.535598,0.613762,"[[937, 10, 27, 17, 3], [86, 142, 53, 125, 68],...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.738190,0.850287,LLama3


In [11]:
robertaDf

,Fold,Train_Losses_Per_Epoch,Val_Losses_Per_Epoch,Balanced_Accuracy,Accuracy,Precision_Micro,Precision_Macro,Precision_Weighted,Recall_Micro,Recall_Macro,Recall_Weighted,F1_Micro,F1_Macro,F1_Weighted,Confusion_Matrix,Val_True_Labels,Val_Predictions,Weighted_Accuracy_0.5,Weighted_Accuracy_1.0,ModelName
0,1,"[2.273344498424704, 0.6563329227557215, 0.3964...","[0.9109003281593323, 0.523340392112732, 0.4629...",0.546540,0.614634,0.614634,0.561393,0.617791,0.614634,0.546540,0.614634,0.614634,0.551343,0.612938,"[[776, 22, 11, 90, 18], [28, 397, 186, 142, 43...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.728755,0.842875,RobertaLarge
1,2,"[2.2537926341394128, 0.6427574809383138, 0.359...","[0.8524569582939148, 0.4681211179494858, 0.401...",0.542944,0.573264,0.573264,0.545999,0.610328,0.573264,0.542944,0.573264,0.573264,0.536701,0.584107,"[[729, 137, 56, 37, 28], [7, 400, 112, 49, 32]...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, ...",0.721399,0.869533,RobertaLarge
2,3,"[2.302797350053037, 0.6249480824122268, 0.3466...","[0.9556651520729065, 0.5182030880451203, 0.462...",0.521130,0.580355,0.580355,0.527020,0.605529,0.580355,0.521130,0.580355,0.580355,0.520926,0.590047,"[[840, 12, 18, 0, 11], [18, 247, 119, 78, 76],...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[4, 4, 4, 1, 3, 3, 4, 3, 1, 3, 3, 1, 3, 4, 1, ...",0.718554,0.856753,RobertaLarge
3,4,"[2.351019258719099, 0.6279265853769416, 0.3830...","[0.8965884923934937, 0.5370388412475586, 0.471...",0.561900,0.591779,0.591779,0.582187,0.632612,0.591779,0.561900,0.591779,0.591779,0.567835,0.606555,"[[792, 162, 26, 95, 17], [11, 190, 12, 340, 55...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.684850,0.777921,RobertaLarge
4,5,"[2.4721066037175854, 0.5970372030118464, 0.338...","[1.0018611645698547, 0.47080618143081665, 0.40...",0.569448,0.637091,0.637091,0.566040,0.637627,0.637091,0.569448,0.637091,0.637091,0.565431,0.634783,"[[946, 14, 24, 10, 0], [65, 175, 39, 104, 91],...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.738940,0.840790,RobertaLarge


In [12]:
mistralDf

,Fold,Flash_Attention,Precision_Macro,Recall_Macro,F1_Macro,Precision_micro,Recall_micro,F1_micro,Precision_weighted,Recall_weighted,...,Balanced_Accuracy,MCC,Cohen_Kappa,Confusion_Matrix,Val_Article_IDs,Val_True_Labels,Val_Predictions,Weighted_Accuracy_0.5,Weighted_Accuracy_1.0,ModelName
0,1,True,0.596051,0.561743,0.568535,0.625417,0.625417,0.625417,0.665317,0.625417,...,0.561743,0.522586,0.518896,"[[768, 9, 31, 96, 13], [15, 374, 214, 96, 97],...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.730552,0.835687,Mistral
1,2,True,0.551455,0.548730,0.536507,0.605387,0.605387,0.605387,0.638654,0.605387,...,0.548730,0.501691,0.495725,"[[724, 191, 38, 24, 10], [3, 455, 96, 40, 6], ...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, ...",0.729553,0.853719,Mistral
2,3,True,0.542132,0.539485,0.534869,0.600273,0.600273,0.600273,0.599504,0.600273,...,0.539485,0.487755,0.486125,"[[831, 19, 22, 4, 5], [21, 277, 102, 112, 26],...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[4, 4, 4, 1, 1, 3, 1, 3, 3, 3, 3, 4, 1, 4, 4, ...",0.721010,0.841746,Mistral
3,4,True,0.554403,0.530380,0.520401,0.591779,0.591779,0.591779,0.581802,0.591779,...,0.530380,0.478604,0.471742,"[[944, 32, 48, 63, 5], [54, 111, 92, 345, 6], ...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",0.702042,0.812306,Mistral
4,5,True,0.564328,0.566713,0.559403,0.634091,0.634091,0.634091,0.630422,0.634091,...,0.566713,0.530551,0.526325,"[[963, 5, 17, 4, 5], [114, 151, 51, 72, 86], [...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.731692,0.829293,Mistral


In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
from matplotlib.lines import Line2D
import ast

# Use the PGF backend
matplotlib.use("pgf")

plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "axes.labelsize": 16,
    "font.size": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

def calculate_dynamic_baselines(dfs):
    all_labels = []
    for df in dfs:
        col_name = 'True_Labels' if 'True_Labels' in df.columns else 'Val_True_Labels'
        for val in df[col_name]:
            if isinstance(val, str):
                all_labels.extend(ast.literal_eval(val))
            else:
                all_labels.extend(val)

    all_labels = np.array(all_labels)
    classes, counts = np.unique(all_labels, return_counts=True)
    probs = counts / len(all_labels)
    num_classes = len(classes)

    macro_base = 1.0 / num_classes

    maj_idx = np.argmax(probs)
    maj_label = classes[maj_idx]
    majority_base_acc = probs[maj_idx]

    tol_10_indices = [i for i, c in enumerate(classes) if abs(c - maj_label) <= 1]
    majority_base_tol10 = np.sum(probs[tol_10_indices])

    majority_base_tol05 = majority_base_acc
    if (maj_label - 1) in classes:
        majority_base_tol05 += 0.5 * probs[np.where(classes == maj_label - 1)[0][0]]
    if (maj_label + 1) in classes:
        majority_base_tol05 += 0.5 * probs[np.where(classes == maj_label + 1)[0][0]]

    return majority_base_acc, macro_base, majority_base_tol05, majority_base_tol10

def plot_core_plus_metrics_boxplot(df1, df2, df3, df4):
    base_maj, base_macro, base_tol05, base_tol10 = calculate_dynamic_baselines([df1, df2, df3, df4])

    combined = pd.concat([df1, df2, df3, df4], ignore_index=True)
    combined['ModelName'] = combined['ModelName'].replace({
        'Bert': 'BERT',
        'RobertaLarge': 'RoBERTa',
        'LLama3': 'Llama',
        'Mistral': 'Mistral',
    })

    metrics = ['Accuracy', 'Precision_Macro', 'Recall_Macro', 'F1_Macro', 'Weighted_Accuracy_0.5', 'Weighted_Accuracy_1.0']
    labels = ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1 Macro', 'Tolerant Acc (0.5)', 'Tolerant Acc (1.0)']
    plot_labels = [l.replace(' ', '\n', 1) if ' ' in l else l for l in labels]

    models = ['BERT', 'RoBERTa', 'Llama', 'Mistral']
    colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
    hatches = ['//', '\\\\', 'xx', '||']
    n_models = len(models)
    width = 0.15
    x = np.arange(len(metrics))

    fig, ax = plt.subplots(figsize=(14, 6))

    for i, model in enumerate(models):
        data = [combined[combined['ModelName'] == model][metric].values for metric in metrics]
        positions = x + i * width
        bp = ax.boxplot(data, positions=positions, widths=width, patch_artist=True, showfliers=False,
                        medianprops=dict(color='black', linewidth=1.5),
                        boxprops=dict(edgecolor='black', linewidth=1))
        for box in bp['boxes']:
            box.set(facecolor=colors[i], hatch=hatches[i], alpha=0.9)

    # --- BASELINES CORRETTE E CENTRATE ---
    offset = width * (n_models - 1) / 2
    centers = x + offset
    group_half = width * n_models / 2 * 0.95 # 0.95 per non toccare i bordi

    c_maj = 'red'
    c_macro = 'darkviolet'

    ax.hlines(y=base_maj, xmin=centers[0]-group_half, xmax=centers[0]+group_half,
              color=c_maj, linestyle='--', linewidth=1.8, zorder=5)
    ax.hlines(y=base_macro, xmin=centers[1]-group_half, xmax=centers[3]+group_half,
              color=c_macro, linestyle='-.', linewidth=1.8, zorder=5)
    ax.hlines(y=base_tol05, xmin=centers[4]-group_half, xmax=centers[4]+group_half,
              color=c_maj, linestyle='--', linewidth=1.8, zorder=5)
    ax.hlines(y=base_tol10, xmin=centers[5]-group_half, xmax=centers[5]+group_half,
              color=c_maj, linestyle='--', linewidth=1.8, zorder=5)

    ax.set_xticks(centers)
    ax.set_xticklabels(plot_labels)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel(r'Classification performance')
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    legend_elements = [
        plt.Rectangle((0, 0), 1, 1, facecolor=colors[i], edgecolor='black', hatch=hatches[i], label=models[i])
        for i in range(n_models)
    ]
    legend_elements.append(Line2D([0], [0], color=c_maj, linestyle='--', label='Majority Class Baseline'))
    legend_elements.append(Line2D([0], [0], color=c_macro, linestyle='-.', label='Random Baseline (1/N)'))
    ax.legend(handles=legend_elements, loc='upper left', frameon=True)

    plt.tight_layout()
    plt.savefig('Results/Plots/results_boxplot_baselines_updated.pdf')

    # --- STAMPA ---
    print("\n" + "="*65)
    print("1. CALCULATED BASELINES")
    print("="*65)
    print(f"{'Metric Group':<25} | {'Type':<15} | {'Value':<8}")
    print("-" * 65)
    print(f"{'Accuracy (Micro)':<25} | {'Majority':<15} | {base_maj:.4f}")
    print(f"{'Macro Metrics (P,R,F1)':<25} | {'Random (1/N)':<15} | {base_macro:.4f}")
    print(f"{'Tolerant Acc (0.5)':<25} | {'Majority':<15} | {base_tol05:.4f}")
    print(f"{'Tolerant Acc (1.0)':<25} | {'Majority':<15} | {base_tol10:.4f}")
    print("="*65)

    print("\n" + "="*65)
    print("2. DETAILED MODEL PERFORMANCE (Mean ± Std Dev across folds)")
    print("="*65)

    for model in models:
        print(f"MODEL: {model}")
        print("-" * 65)
        print(f"{'Metric':<25} | {'Mean':<10} | {'Std Dev':<10}")
        print("-" * 50)
        model_data = combined[combined['ModelName'] == model]
        for metric, label in zip(metrics, labels):
            mean_val = model_data[metric].mean()
            std_val = model_data[metric].std()
            print(f"{label:<25} | {mean_val:.4f} | {std_val:.4f}")
        print("="*65 + "\n")

plot_core_plus_metrics_boxplot(bertDf, llamaDf, robertaDf, mistralDf)


1. CALCULATED BASELINES
Metric Group              | Type            | Value   
-----------------------------------------------------------------
Accuracy (Micro)          | Majority        | 0.2588
Macro Metrics (P,R,F1)    | Random (1/N)    | 0.2000
Tolerant Acc (0.5)        | Majority        | 0.4269
Tolerant Acc (1.0)        | Majority        | 0.5950

2. DETAILED MODEL PERFORMANCE (Mean ± Std Dev across folds)
MODEL: BERT
-----------------------------------------------------------------
Metric                    | Mean       | Std Dev   
--------------------------------------------------
Accuracy                  | 0.5831 | 0.0277
Precision (Macro)         | 0.5278 | 0.0221
Recall (Macro)            | 0.5164 | 0.0195
F1 Macro                  | 0.5174 | 0.0203
Tolerant Acc (0.5)        | 0.7021 | 0.0306
Tolerant Acc (1.0)        | 0.8211 | 0.0392

MODEL: RoBERTa
-----------------------------------------------------------------
Metric                    | Mean       | Std Dev   
--

In [14]:
import numpy as np
import ast

metrics = ['Accuracy', 'Precision_Macro', 'Recall_Macro', 'F1_Macro', 'Weighted_Accuracy_0.5', 'Weighted_Accuracy_1.0']

print("="*70)
print("COLUMN CHECK")
print("="*70)
for name, df in [('bertDf', bertDf), ('llamaDf', llamaDf), ('robertaDf', robertaDf), ('mistralDf', mistralDf)]:
    print(f"\n{name}:")
    print(f"  ModelName values : {df['ModelName'].unique()}")
    label_col = 'True_Labels' if 'True_Labels' in df.columns else 'Val_True_Labels'
    print(f"  Label column     : {label_col}")
    missing = [m for m in metrics if m not in df.columns]
    print(f"  Missing metrics  : {missing if missing else 'None'}")
    nulls = df[metrics].isnull().sum()
    if nulls.any():
        print(f"  NaN in metrics   : {nulls[nulls > 0].to_dict()}")
    else:
        print(f"  NaN in metrics   : None")

print("\n" + "="*70)
print("HATCH RENDER TEST (PGF NaN trigger)")
print("="*70)
import pandas as pd
combined = pd.concat([bertDf, llamaDf, robertaDf, mistralDf], ignore_index=True)
combined['ModelName'] = combined['ModelName'].replace({
    'Bert': 'BERT', 'RobertaLarge': 'RoBERTa', 'LLama3': 'Llama', 'Mistral': 'Mistral'
})
models  = ['BERT', 'RoBERTa', 'Llama', 'Mistral']
for model in models:
    subset = combined[combined['ModelName'] == model]
    print(f"\n{model}: {len(subset)} rows")
    for m in metrics:
        vals = subset[m].values
        nan_count = np.sum(np.isnan(vals.astype(float)))
        print(f"  {m:<30} n={len(vals)}  NaN={nan_count}  min={np.nanmin(vals):.4f}  max={np.nanmax(vals):.4f}")

print("\n" + "="*70)
print("BASELINE LABEL PARSE TEST")
print("="*70)
for name, df in [('bertDf', bertDf), ('llamaDf', llamaDf), ('robertaDf', robertaDf), ('mistralDf', mistralDf)]:
    col = 'True_Labels' if 'True_Labels' in df.columns else 'Val_True_Labels'
    try:
        all_labels = []
        for val in df[col]:
            if isinstance(val, str):
                all_labels.extend(ast.literal_eval(val))
            else:
                all_labels.extend(val)
        arr = np.array(all_labels)
        print(f"{name}: OK — {len(arr)} labels, classes={np.unique(arr).tolist()}")
    except Exception as e:
        print(f"{name}: ERROR — {e}")

COLUMN CHECK

bertDf:
  ModelName values : ['Bert']
  Label column     : True_Labels
  Missing metrics  : None
  NaN in metrics   : None

llamaDf:
  ModelName values : ['LLama3']
  Label column     : Val_True_Labels
  Missing metrics  : None
  NaN in metrics   : None

robertaDf:
  ModelName values : ['RobertaLarge']
  Label column     : Val_True_Labels
  Missing metrics  : None
  NaN in metrics   : None

mistralDf:
  ModelName values : ['Mistral']
  Label column     : Val_True_Labels
  Missing metrics  : None
  NaN in metrics   : None

HATCH RENDER TEST (PGF NaN trigger)

BERT: 5 rows
  Accuracy                       n=5  NaN=0  min=0.5473  max=0.6166
  Precision_Macro                n=5  NaN=0  min=0.4935  max=0.5484
  Recall_Macro                   n=5  NaN=0  min=0.4875  max=0.5369
  F1_Macro                       n=5  NaN=0  min=0.4878  max=0.5371
  Weighted_Accuracy_0.5          n=5  NaN=0  min=0.6708  max=0.7349
  Weighted_Accuracy_1.0          n=5  NaN=0  min=0.7655  max=0.8652


# Weighted Accuracy metric

In [15]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

# -----------------------
# Data & Settings
# -----------------------
conf_matrix = np.array([
    [25, 5, 3, 1, 1],
    [4, 20, 6, 3, 1],
    [2, 4, 15, 5, 4],
    [1, 2, 5, 18, 4],
    [0, 1, 2, 6, 21]
])

num_classes = conf_matrix.shape[0]
adjacent_weight = 0.5
adjacent_cells = [(i, j) for i in range(num_classes) for j in range(num_classes) if abs(i - j) == 1]

correct = np.trace(conf_matrix)
adjacent_sum = sum(conf_matrix[i, j] for i, j in adjacent_cells)
total = conf_matrix.sum()

accuracy_raw = correct / total
weighted_acc_half = (correct + 0.5 * adjacent_sum) / total
weighted_acc_full = (correct + adjacent_sum) / total

weighted_num_half = correct + 0.5 * adjacent_sum
weighted_num_full = correct + adjacent_sum

# -----------------------
# Helpers
# -----------------------
def fmt_num(x):
    return f"{x:.1f}" if x % 1 else str(int(x))

def format_cell_value(value, weight=1):
    weighted_val = value * weight
    return f"{weighted_val:.1f}" if weighted_val != int(weighted_val) else str(int(weighted_val))

# -----------------------
# Annotations
# -----------------------
annot_left = np.vectorize(lambda x: fmt_num(x))(conf_matrix)
annot_center = np.empty_like(conf_matrix, dtype=object)
annot_right = np.vectorize(lambda x: fmt_num(x))(conf_matrix)

for i in range(num_classes):
    for j in range(num_classes):
        if (i, j) in adjacent_cells:
            annot_center[i, j] = format_cell_value(conf_matrix[i, j], weight=adjacent_weight)
        else:
            annot_center[i, j] = fmt_num(conf_matrix[i, j])

# -----------------------
# Plotting Function
# -----------------------
def plot_matrix(title, annotations, cmap, formula_text, final_value, bbox_color, filename, x_formula_center=1, draw_rectangles=True):
    fig, ax = plt.subplots(figsize=(6, 6))

    sns.heatmap(
        conf_matrix,
        annot=annotations,
        fmt='',
        cmap=cmap,
        ax=ax,
        cbar=False,
        annot_kws={"size": 16}
    )

    ax.set_xlabel("Predicted", fontsize=14)
    ax.set_ylabel("True", fontsize=14)
    ax.set_title(title, fontsize=16, pad=20)
    
    # Corrected lines to set the tick labels to start from 1
    tick_labels = [str(i + 1) for i in range(num_classes)]
    ax.set_xticklabels(tick_labels, fontsize=12)
    ax.set_yticklabels(tick_labels, fontsize=12, rotation=0)

    # Draw rectangles for adjacent cells
    if draw_rectangles:
        for (i, j) in adjacent_cells:
            ax.add_patch(Rectangle((j, i), 1, 1, fill=False, edgecolor='red', lw=2))

    # --- Formula Below ---
    formula_fontsize = 16
    y_formula = 0.02
    char_width = 0.004  # controls spacing – tweak if needed

    text_before = formula_text
    value = final_value

    total_chars = len(text_before) + len(value)
    total_width = total_chars * char_width
    x_start = x_formula_center - total_width / 2
    x_split = x_start + len(text_before) * char_width

    bbox = dict(boxstyle='round,pad=0.2', facecolor=bbox_color[0], edgecolor=bbox_color[1])

    fig.text(x_split, y_formula, text_before, fontsize=formula_fontsize, ha='right', va='center')
    fig.text(x_split, y_formula, f'$\\bf{{{value}}}$', fontsize=formula_fontsize+1, ha='left', va='center', bbox=bbox)

    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    plt.savefig(filename, dpi=300)
    plt.show()

# -----------------------
# Plot Each Matrix
# -----------------------

# Standard Accuracy
plot_matrix(
    title="Standard Accuracy",
    annotations=annot_left,
    cmap='Blues',
    formula_text=f"= ({correct}) / ({total}) = ",
    final_value=f"{accuracy_raw:.3f}",
    bbox_color=('#cceeff', '#007bff'),
    filename='standard_accuracy.png',
    x_formula_center=0.55,
    draw_rectangles=False
)
plt.savefig('Results/Plots/standard_accuracy')


# Weighted Accuracy (0.5)
plot_matrix(
    title="Tolerant Accuracy (Adjacent = 0.5)",
    annotations=annot_left,
    cmap='YlGn',
    formula_text=f"= ({correct} + 0.5 × {adjacent_sum}) / ({total}) = {weighted_num_half:.1f} / {total} = ",
    final_value=f"{weighted_acc_half:.3f}",
    bbox_color=('#d4edda', '#28a745'),
    filename='tolernat_05_accuracy.png',
    x_formula_center=0.8
)
plt.savefig('Results/Plots/tolernat_05_accuracy')



# Weighted Accuracy (1.0)
plot_matrix(
    title="Tolerant Accuracy (Adjacent = 1.0)",
    annotations=annot_right,
    cmap='OrRd',
    formula_text=f"= ({correct} + {adjacent_sum}) / ({total}) = {weighted_num_full} / {total} = ",
    final_value=f"{weighted_acc_full:.3f}",
    bbox_color=('#f8d7da', '#dc3545'),
    filename='tolernat_10_accuracy.png',
    x_formula_center=0.75
)

plt.savefig('Results/Plots/tolernat_10_accuracy')



/tmp/ipykernel_277515/3285702020.py:106: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_277515/3285702020.py:106: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_277515/3285702020.py:106: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
import re
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. CONFIGURATION (STRICTLY AS REQUESTED)
# ==========================================
# Use the PGF backend
matplotlib.use("pgf")

# Configure LaTeX rendering
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": [],
    "axes.labelsize": 16,
    "font.size": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "pgf.rcfonts": False,
})

# ==========================================
# 2. DATA LOADING
# ==========================================
log_filename = 'Results/Classification/finetuning_run.log'

try:
    with open(log_filename, 'r', encoding='utf-8') as f:
        log_content = f.read()
except FileNotFoundError:
    print(f"Error: File '{log_filename}' not found. Please check the path.")
    exit()

train_losses = []
val_losses = []
epochs = []

# Pattern per estrarre i risultati (Manteniamo la regex in Italiano per leggere il log)
pattern = r'RISULTATI EPOCA (\d+): Train Loss: ([\d.]+) \| Val Loss: ([\d.]+)'
matches = re.findall(pattern, log_content)

for match in matches:
    epoch = int(match[0])
    train_loss = float(match[1])
    val_loss = float(match[2])
    
    epochs.append(epoch)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

if not epochs:
    print("No data found in log file. Check regex pattern.")
    exit()

# ==========================================
# 3. PLOTTING (ENGLISH TRANSLATION)
# ==========================================
plt.figure(figsize=(10, 6)) # Adjusted slightly for standard LaTeX text width

# Plot losses
plt.plot(epochs, train_losses, 'b-o', label='Training Loss', linewidth=2, markersize=6)
plt.plot(epochs, val_losses, 'r-s', label='Validation Loss', linewidth=2, markersize=6)

# Find minimum validation loss
min_val_idx = np.argmin(val_losses)
min_val_epoch = epochs[min_val_idx]
min_val_loss = val_losses[min_val_idx]

# Highlight minimum point
plt.axvline(x=min_val_epoch, color='g', linestyle='--', alpha=0.7)
plt.plot(min_val_epoch, min_val_loss, 'g*', markersize=18, 
         label=f'Best Model (Epoch {min_val_epoch})')

# Graph Configuration (Labels in English)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(fontsize=10, loc='upper right') # Added location for stability
plt.grid(True, alpha=0.3)
plt.xticks(epochs)

# Annotations (English)
# Moving the text slightly to avoid overlap depending on the curve shape
plt.text(min_val_epoch, min_val_loss + (max(val_losses)*0.02), 
         f'  Epoch {min_val_epoch}\n  Loss: {min_val_loss:.4f}',
         fontsize=10, verticalalignment='bottom', color='green')

plt.tight_layout()

# Save as PDF
output_filename = 'Results/Plots/loss_plot.pdf'
plt.savefig(output_filename, dpi=300, bbox_inches='tight')
print(f"Graph saved successfully as: {output_filename}")

# ==========================================
# 4. STATISTICAL ANALYSIS (OPTIONAL TRANSLATION)
# ==========================================
print("\n" + "=" * 60)
print("LOSS TREND ANALYSIS")
print("=" * 60)
print(f"Total epochs: {len(epochs)}")
print(f"Best Epoch (Min Val Loss): {min_val_epoch}")
print(f"Min Val Loss: {min_val_loss:.4f}")
print(f"Final Train Loss (Epoch {epochs[-1]}): {train_losses[-1]:.4f}")
print(f"Final Val Loss (Epoch {epochs[-1]}): {val_losses[-1]:.4f}")

# Overfitting check
overfitting_start = None
for i in range(min_val_idx + 1, len(val_losses)):
    if val_losses[i] > val_losses[min_val_idx]:
        overfitting_start = epochs[i]
        break

if overfitting_start:
    print(f"\nOverfitting likely started at epoch: {overfitting_start}")
else:
    print("\nNo clear sign of overfitting detected.")

Graph saved successfully as: Results/Plots/loss_plot.pdf

LOSS TREND ANALYSIS
Total epochs: 20
Best Epoch (Min Val Loss): 4
Min Val Loss: 0.4416
Final Train Loss (Epoch 20): 0.0060
Final Val Loss (Epoch 20): 0.6863

Overfitting likely started at epoch: 5


In [17]:
import matplotlib.pyplot as plt
import numpy as np
import re
import os
from matplotlib.colors import LinearSegmentedColormap
def plot_confusion_matrices(df, figsize=(15, 10), filename = 'noName'):
    """
    Plot confusion matrices in a grid layout with average matrix and save to Results/CM
    Args:
        df (pd.DataFrame): DataFrame containing 'Confusion_Matrix' column
        figsize (tuple): Figure size (width, height)
        include_average (bool): Whether to include average confusion matrix
    """
    include_average=True
    # Create save directory
    save_dir = "Results/CM"
    os.makedirs(save_dir, exist_ok=True)
    # Parse all confusion matrices
    confusions_list = []
    n_classes = None
    for confusion_str in df['Confusion_Matrix']:
        numbers = re.findall(r'\d+', confusion_str)
        cm_flat = list(map(int, numbers))
        n = int(np.sqrt(len(cm_flat)))
        cm = np.array(cm_flat).reshape(n, n)
        confusions_list.append(cm)
        if n_classes is None:
            n_classes = n
        elif n != n_classes:
            raise ValueError("Inconsistent matrix dimensions across folds")
    # Compute average matrix (normalized)
    if include_average:
        avg_cm = np.mean(confusions_list, axis=0)
        norm_avg_cm = avg_cm / np.maximum(1, avg_cm.sum(axis=1, keepdims=True))
        total_plots = len(confusions_list) + 1
    else:
        total_plots = len(confusions_list)
    # Setup grid
    cols = int(np.ceil(np.sqrt(total_plots)))
    rows = int(np.ceil(total_plots / cols))
    # Create color map
    colors = ["#f7fbff", "#deebf7", "#c6dbef", "#9ecae1",
              "#6baed6", "#4292c6", "#2171b5", "#08519c", "#08306b"]
    cmap = LinearSegmentedColormap.from_list("custom_blues", colors)
    # Create figure
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    plt.subplots_adjust(wspace=0.5, hspace=0.5)
    axes = axes.ravel() if isinstance(axes, np.ndarray) else [axes]
    # Plot and save individual matrices
    for idx, cm in enumerate(confusions_list):
        ax = axes[idx]
        title = f'Fold {idx+1}'
        _plot_single_cm(ax, cm, title, cmap, False)
        # Save individual matrix without title
        fig_single, ax_single = plt.subplots(figsize=(8, 6))
        _plot_single_cm(ax_single, cm, "", cmap, False)
        plt.savefig(os.path.join(save_dir, f'fold_{idx+1}_cm.png'),
                    bbox_inches='tight', dpi=100)
        plt.close(fig_single)
    # Plot and save average matrix
    if include_average:
        ax_avg = axes[len(confusions_list)]
        _plot_single_cm(ax_avg, norm_avg_cm, 'Average (Normalized)', cmap, True)
        # Save average matrix without title
        fig_avg, ax_avg_save = plt.subplots(figsize=(8, 6))
        _plot_single_cm(ax_avg_save, norm_avg_cm, "", cmap, True)
        plt.savefig(os.path.join(save_dir, 'average_cm.png'),
                    bbox_inches='tight', dpi=100)
        plt.close(fig_avg)
    # Hide empty subplots
    for ax in axes[total_plots:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.savefig('Results/Plots/cm_' + filename)




def _plot_single_cm(ax, cm, title, cmap, is_normalized):
    """Helper to plot single confusion matrix with custom mapping and LaTeX support"""
    
    # Mapping delle etichette (da 0-4 a l1-l5)
    labelMapping = {
        0: r"\textit{l1}",
        1: r"\textit{l2}",
        2: r"\textit{l3}",
        3: r"\textit{l4}",
        4: r"\textit{l5}"
    }
    
    # Plot matrix
    vmax = 1.0 if is_normalized else cm.max()
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap, vmin=0, vmax=vmax)
    
    # Titolo (leggermente più grande)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    
    # Annotazioni numeriche all'interno delle celle
    threshold = vmax / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm[i, j]
            text = f"{val:.2f}" if is_normalized else str(int(val))
            ax.text(j, i, text,
                    ha="center", va="center",
                    color="white" if val > threshold else "black",
                    fontsize=14) # Dimensione numeri interna

    # Configurazione Etichette Assi
    ax.set_xlabel('Predicted Label', labelpad=15, fontsize=16)
    ax.set_ylabel('True Label', labelpad=15, fontsize=16)
    
    # Applichiamo il mapping alle etichette (Tick Labels)
    tick_marks = np.arange(cm.shape[0])
    labels = [labelMapping.get(i, str(i)) for i in tick_marks]
    
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    
    # Rotazione opzionale per evitare sovrapposizioni se le etichette sono lunghe
    ax.set_xticklabels(labels, fontsize=12, rotation=0, ha="right")
    ax.set_yticklabels(labels, fontsize=12)
    
    # Grid e stile dei bordi
    ax.set_xticks(np.arange(cm.shape[0])-0.5, minor=True)
    ax.set_yticks(np.arange(cm.shape[0])-0.5, minor=True)
    ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

In [18]:
import pandas as pd

# i tuoi 4 file - usa gli stessi path del codice precedente
model_files = {
    "bert":    "Results/Classification/bert_enriched.csv",
    "roberta": "Results/Classification/roberta_enriched.csv",
    "llama":   "Results/Classification/llama_enriched.csv",
    "mistral": "Results/Classification/mistral/24B_results_enriched.csv"
}

for name, path in model_files.items():
    print(f"\n=== {name.upper()} ===")
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        print(f"File non trovato: {path}")
        continue

    # la funzione si aspetta questa colonna
    if "Confusion_Matrix" not in df.columns:
        print(f"ATTENZIONE: {name} non ha la colonna 'Confusion_Matrix'. Salto.")
        continue

    # chiama la tua funzione - filename serve per il pdf finale
    plot_confusion_matrices(df, figsize=(15, 10), filename=name)


=== BERT ===


/tmp/ipykernel_277515/2952417826.py:74: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()



=== ROBERTA ===


/tmp/ipykernel_277515/2952417826.py:74: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()



=== LLAMA ===


/tmp/ipykernel_277515/2952417826.py:74: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()



=== MISTRAL ===


/tmp/ipykernel_277515/2952417826.py:74: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
